<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/StructureDistance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install --force-reinstall 'sqlalchemy<2.0.0'

In [ ]:
import os
!pwd
os.chdir('/content')
# Clone ALIGNN repo to get example folder
if not os.path.exists('graphdot'):
  !git clone https://gitlab.com/knc6/graphdot.git

os.chdir('graphdot')
# Install using setup.py in case pip didn't work
# !python setup.py devel!op
!pip install -e .
#!pip install dgl-cu111 # Colab has cuda 11.1

In [ ]:
pip install ase

In [ ]:
pip install pymatgen

In [ ]:
import numpy as np
import pandas as pd
from ase.build import molecule, bulk
from graphdot import Graph
from graphdot.kernel.molecular import Tang2019MolecularKernel

# build sample molecules
small_title = ['H2O', 'HCl', 'NaCl']
bulk_title = ['NaCl-bulk', 'NaCl-bulk2']
bulk = [
    bulk('NaCl', 'rocksalt', a=5.64),
    bulk('NaCl', 'rocksalt', a=5.66),
]
molecules = [molecule(name) for name in small_title] + bulk

# convert to molecular graphs
graphs = [Graph.from_ase(m) for m in molecules]

# use pre-defined molecular kernel
kernel = Tang2019MolecularKernel(edge_length_scale=0.1)

R = kernel(graphs)

# normalize the similarity matrix
d = np.diag(R)**-0.5
K = np.diag(d).dot(R).dot(np.diag(d))

# note the difference between the NaCl variants
title = small_title + bulk_title
print(pd.DataFrame(K, columns=title, index=title))

In [ ]:
pip install alignn jarvis-tools

In [ ]:

from jarvis.db.figshare import data
import pandas as pd
from jarvis.core.atoms import Atoms

dft_3d = data('dft_3d')
df=pd.DataFrame(dft_3d)

In [ ]:
si=['JVASP-1002','JVASP-91933','JVASP-25369']#,'JVASP-25368']
mos2=['JVASP-28733','JVASP-28413','JVASP-58505']
ni3al=['JVASP-14971','JVASP-99749','JVASP-11979']
sio2=['JVASP-58349', 'JVASP-34674','JVASP-34656']

In [ ]:
def atoms_jid(ids=[],verbose=True):
  get_atoms=[]
  for i in ids:
    atoms=(Atoms.from_dict(df[df['jid']==i].atoms.values[0]))
    if verbose:
        print(i,atoms.composition.reduced_formula,atoms.get_spacegroup)
    ase_atoms=atoms.ase_converter()
    get_atoms.append(ase_atoms)
  return get_atoms

def get_df(ids=[],verbose=True):
    ase_atoms_list = atoms_jid(ids,verbose=verbose)
    graphs = []
    ids_list = []
    for i,j in zip (ids,ase_atoms_list):
      try:
          graphs.append(Graph.from_ase(j) )
          #ids_list.append(i)
          ids_list.append(j.get_chemical_formula())
      except Exception as exp:
        print('exp',exp,i)
        pass

    # graphs = [Graph.from_ase(m) for m in ase_atoms_list]

    # use pre-defined molecular kernel
    kernel = Tang2019MolecularKernel(edge_length_scale=0.1)

    R = kernel(graphs)

    # normalize the similarity matrix
    d = np.diag(R)**-0.5
    K = np.diag(d).dot(R).dot(np.diag(d))

    # note the difference between the NaCl variants

    df_kernel = (pd.DataFrame(K, columns=ids_list, index=ids_list))
    return df_kernel

# si_atoms = atoms_jid(si)

In [ ]:
get_df(si)

In [ ]:
get_df(ni3al)

In [ ]:
get_df(mos2)

In [ ]:
get_df(sio2)

In [ ]:
x = get_df(si+sio2+ni3al+mos2)

In [ ]:
x

In [ ]:
import seaborn as sns
cm = sns.light_palette("blue", as_cmap=True)
# x=x.style.background_gradient(cmap=cm)
# display(x)

In [ ]:
%%time
todo_jids = df.jid.values[0:10]
x = get_df(todo_jids)
import seaborn as sns
cm = sns.light_palette("blue", as_cmap=True)
x=x.style.background_gradient(cmap=cm)
display(x)

In [ ]:
%%time
todo_jids = df.jid.values[0:100]
x = get_df(todo_jids)
import seaborn as sns
cm = sns.light_palette("blue", as_cmap=True)
x=x.style.background_gradient(cmap=cm)
display(x)

In [ ]:
%%time
todo_jids = df.jid.values[0:1000]
x = get_df(todo_jids)
x
# import seaborn as sns
# cm = sns.light_palette("blue", as_cmap=True)
# x=x.style.background_gradient(cmap=cm)
# display(x)

In [ ]:
%%time
todo_jids = df.jid.values[0:10000]
x = get_df(todo_jids)
import seaborn as sns
cm = sns.light_palette("blue", as_cmap=True)
x=x.style.background_gradient(cmap=cm)
display(x)

In [ ]:
%%time
todo_jids = df.jid.values #[0:100]
x = get_df(todo_jids)
import seaborn as sns
cm = sns.light_palette("blue", as_cmap=True)
x=x.style.background_gradient(cmap=cm)
display(x)

In [ ]:
!pip install -q dgl==1.0.1+cu117 -f https://data.dgl.ai/wheels/cu117/repo.html
!pip install -q alignn

In [ ]:
from jarvis.core.atoms import Atoms
from jarvis.core.graphs import Graph
from alignn.pretrained import get_figshare_model
import torch
model = get_figshare_model()
device = "cpu"
if torch.cuda.is_available():
    device = torch.device("cuda")
model=model.to(device)
box = [[2.715, 2.715, 0], [0, 2.715, 2.715], [2.715, 0, 2.715]]
coords = [[0, 0, 0], [0.25, 0.25, 0.25]]
elements = ["Si", "Si"]
atoms = Atoms(lattice_mat=box, coords=coords, elements=elements)

def get_val(model,g,lg):
    activation = {}
    def getActivation(name):
        # the hook signature
        def hook(model, input, output):
            activation[name] = output.detach()
        return hook
    h = model.readout.register_forward_hook(getActivation('readout'))
    g=g.to(device)
    lg=lg.to(device)
    out = model([g,lg])
    h.remove()
    return activation['readout'][0]

def get_alignn_feats(atoms=[],model):
    activation = {}
    def getActivation(name):
        # the hook signature
        def hook(model, input, output):
            activation[name] = output.detach()
        return hook
    h = model.readout.register_forward_hook(getActivation('readout'))
    g=g.to(device)
    lg=lg.to(device)
    out = model([g,lg])
    h.remove()
    return activation['readout'][0]

g,lg=Graph.atom_dgl_multigraph(atoms)
feats = get_val(model,g,lg)

In [ ]:
feats